In [ ]:
import os
import time
import pandas as pd
import geopandas as gpd
import numpy as np
import libpysal
from esda.getisord import G_Local
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
BASE_DIR = Path.cwd()
PROJECT_ROOT = BASE_DIR.parent if BASE_DIR.name == "notebooks" else BASE_DIR

# --- Grid geometry (Step 5 output) ---
grid_path = os.path.join(PROJECT_ROOT, "data", "raw", "ihr_grid_2_5km_wgs84.gpkg")
grid = gpd.read_file(grid_path)  # columns: Cell_ID, cell_size_km, geometry

# --- Cell x Year wide table (Step 7 output) ---
wide_path = os.path.join(PROJECT_ROOT, "data", "raw", "ihr_cell_year_wide.csv")
wide = pd.read_csv(wide_path)  # columns: Cell_ID, 2001, 2002, ..., 2025

year_cols = [c for c in wide.columns if c != "Cell_ID"]
print(f"Grid cells: {len(grid):,} | Wide table cells: {len(wide):,} | Years: {len(year_cols)}")

In [ ]:
# Align the two datasets to a common set of Cell_IDs (defensive - they should
# already match, but a mismatch here would silently corrupt every year's Gi*).
common_ids = set(grid["Cell_ID"]) & set(wide["Cell_ID"])
dropped_from_grid = len(grid) - len(common_ids)
dropped_from_wide = len(wide) - len(common_ids)

if dropped_from_grid or dropped_from_wide:
    print(f"Warning: dropping {dropped_from_grid} grid-only cells and "
          f"{dropped_from_wide} wide-table-only cells before analysis.")

grid_aligned = grid[grid["Cell_ID"].isin(common_ids)].reset_index(drop=True)
wide_aligned = wide[wide["Cell_ID"].isin(common_ids)].set_index("Cell_ID")

print(f"Aligned dataset: {len(grid_aligned):,} cells x {len(year_cols)} years")

# Stage 8: Spatial Weights

Building Queen weights over ~90K polygons is a one-time, moderately expensive step, so the result
is cached to disk (`data/processed/ihr_queen_weights.gal`) and reloaded on subsequent runs instead
of being rebuilt every time.

For a proper Getis-Ord **Gi\*** (as opposed to Gi), each cell must be included in its own
neighbourhood. Rather than letting `esda` silently infer a self-weight (which raises a warning
and is ambiguous once weights are row-standardized), the diagonal is filled explicitly with
`libpysal.weights.fill_diagonal()` before row-standardizing.


In [ ]:
weights_dir = os.path.join(PROJECT_ROOT, "data", "processed")
os.makedirs(weights_dir, exist_ok=True)
weights_path = os.path.join(weights_dir, "ihr_queen_weights.gal")

if os.path.exists(weights_path):
    print(f"Loading cached spatial weights from {weights_path}")
    w = libpysal.io.open(weights_path, "r").read()
else:
    print("Building Queen contiguity weights")
    t0 = time.time()
    w = libpysal.weights.Queen.from_dataframe(
        grid_aligned, ids=grid_aligned["Cell_ID"].tolist()
    )
    print(f"Built in {time.time() - t0:.1f}s")

    f = libpysal.io.open(weights_path, "w")
    f.write(w)
    f.close()
    print(f"Cached to {weights_path}")

# Islands = cells with no Queen neighbours at all (isolated polygons, e.g. slivers
# or disconnected fragments). Their Gi* will reflect only their own value once the
# diagonal is filled, which is not a meaningful "neighbourhood" statistic.
islands = list(w.islands)
print(f"Islands (zero-neighbour cells): {len(islands)}")
if islands[:10]:
    print("Sample island Cell_IDs:", islands[:10])

# Include each cell in its own neighbourhood (required for Gi*, not Gi), then
# row-standardize so weights per row sum to 1.
w = libpysal.weights.fill_diagonal(w, val=1.0)
w.transform = "r"

print(f"Spatial weights ready: {w.n:,} cells, mean neighbours/cell = {w.mean_neighbors:.2f}")

In [ ]:
# Size of each connected component
component_sizes = pd.Series(w.component_labels).value_counts().sort_index()

print("Connected components:", len(component_sizes))
print("\nComponent sizes:")
print(component_sizes)

## Step 9 - Getis-Ord Gi* per year

Loops over each year, computes Gi* for every cell using the fixed weights matrix above, and writes
one CSV per year to `outputs/gi_star_annual/`. Already-completed years are skipped automatically,
so this cell is safe to re-run if interrupted partway through.

**Efficiency note:** `PERMUTATIONS` controls the conditional-permutation inference (`Gi_star_p_sim`).
At ~90K cells x 25 years, a high permutation count multiplies runtime substantially. Start with a
small value (or 0, relying only on the analytical `Gi_star_p_norm`) to gauge per-year timing before
committing to a full run with more permutations.

In [ ]:
PERMUTATIONS = 999  # lower this (or set to 0) if per-year runtime is too slow - see note above

output_dir = os.path.join(PROJECT_ROOT, "outputs", "gi_star_annual")
os.makedirs(output_dir, exist_ok=True)

for year in year_cols:
    out_path = os.path.join(output_dir, f"ihr_gi_{year}.csv")
    if os.path.exists(out_path):
        print(f"{year}: already exists, skipping ({out_path})")
        continue

    t0 = time.time()
    y = wide_aligned.loc[w.id_order, year].values

    gi = G_Local(y, w, star=None, permutations=PERMUTATIONS)

    year_df = pd.DataFrame({
        "Cell_ID": w.id_order,
        "Year": int(year),
        "Gi_star_z": gi.Zs,
        "Gi_star_p_sim": gi.p_sim,
        "Gi_star_p_norm": gi.p_norm,
    })
    year_df.to_csv(out_path, index=False)

    print(f"{year}: {len(year_df):,} cells written to {out_path} ({time.time() - t0:.1f}s)")

print("\nStep 9 complete.")

## Combine into a single long-format Gi* table

Stacks all per-year CSVs into one `Cell_ID | Year | Gi_star_z | Gi_star_p_sim | Gi_star_p_norm`
table - the natural input for Step 10 (Mann-Kendall on each cell's Gi* time series).

In [ ]:
csv_files = sorted(Path(output_dir).glob("ihr_gi_*.csv"))

gi_long = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

n_dupes = gi_long.duplicated(subset=["Cell_ID", "Year"]).sum()
if n_dupes:
    raise ValueError(f"Found {n_dupes} duplicate (Cell_ID, Year) pairs - check per-year CSVs.")

print(f"Combined Gi* table: {len(gi_long):,} rows "
      f"({gi_long['Cell_ID'].nunique():,} cells x {gi_long['Year'].nunique()} years)")

combined_path = os.path.join(PROJECT_ROOT, "outputs", "ihr_gi_star_2001_2025.csv")
gi_long.to_csv(combined_path, index=False)
print(f"Saved -> {combined_path}")

gi_long.head()

## Quick QA

In [ ]:
print("Rows with NaN Gi_star_z:", gi_long["Gi_star_z"].isna().sum())
print("Islands present in output:", gi_long["Cell_ID"].isin(islands).sum(), "rows")

print("\nGi_star_z summary across all cells/years:")
print(gi_long["Gi_star_z"].describe())

# Cells consistently at the extremes (quick eyeball check, not a formal classification)
mean_z_by_cell = gi_long.groupby("Cell_ID")["Gi_star_z"].mean().sort_values(ascending=False)
print("\nTop 5 cells by mean Gi_star_z across all years:")
print(mean_z_by_cell.head())

In [ ]:
# Map one year's Gi* z-scores as a sanity check
# sample_year = int(year_cols[0])
sample_year = 2025

plot_df = grid_aligned.merge(
    gi_long[gi_long["Year"] == sample_year],
    on="Cell_ID",
    how="left",
)

ax = plot_df.plot(
    column="Gi_star_z",
    cmap="RdBu_r",
    legend=True,
    figsize=(10, 10),
    vmin=-3, vmax=3,
)
ax.set_title(f"Getis-Ord Gi* z-scores - {sample_year}")
ax.set_axis_off()

### Verify y actually varies across cells for 2001

## Verify star=True-equivalent behavior (self-weight correctly included)

### Check the self-weight is actually present and correctly normalized

### Inspect the distribution of 2001 Gi* values

In [ ]:
gi_2001 = gi_long[gi_long["Year"] == 2001]

print(gi_2001["Gi_star_z"].describe())

# Flag suspiciously large tied blocks (a handful of ties is expected at grid edges;
# thousands sharing one exact value would not be)
tie_counts = gi_2001["Gi_star_z"].value_counts()
print("\nLargest tied groups:")
print(tie_counts.head(10))

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(gi_2001["Gi_star_z"], bins=100)
axes[0].set_title("Gi* z-score distribution, 2001 (full range)")
axes[1].hist(gi_2001["Gi_star_z"], bins=100, range=(-3, 5))
axes[1].set_title("Gi* z-score distribution, 2001 (zoomed)")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 2001 Gi* SIGNIFICANCE THRESHOLDS
# ============================================================

z = gi_2001["Gi_star_z"]

thresholds = [
    ("High cluster",  +1.96),
    ("High cluster",  +2.58),
    ("High cluster",  +3.29),
    ("Low cluster",   -1.96),
    ("Low cluster",   -2.58),
    ("Low cluster",   -3.29),
]

n = len(z)

print(f"Total cells: {n:,}\n")

for label, threshold in thresholds:

    if threshold > 0:
        count = (z >= threshold).sum()
        condition = f"Gi* ≥ +{threshold:.2f}"
    else:
        count = (z <= threshold).sum()
        condition = f"Gi* ≤ {threshold:.2f}"

    pct = 100 * count / n

    print(
        f"{label:14s} | "
        f"{condition:14s} | "
        f"{count:6,} cells | "
        f"{pct:6.3f}%"
    )

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

z = gi_2001["Gi_star_z"].dropna().to_numpy()

plt.figure(figsize=(12, 6))

plt.hist(z, bins=150)

# Positive thresholds
plt.axvline(1.96, linestyle="--", linewidth=1.5, label="+1.96")
plt.axvline(2.58, linestyle="--", linewidth=1.5, label="+2.58")
plt.axvline(3.29, linestyle="--", linewidth=1.5, label="+3.29")

# Negative thresholds
plt.axvline(-1.96, linestyle="--", linewidth=1.5, label="-1.96")
plt.axvline(-2.58, linestyle="--", linewidth=1.5, label="-2.58")
plt.axvline(-3.29, linestyle="--", linewidth=1.5, label="-3.29")

plt.xlabel("Gi* z-score")
plt.ylabel("Number of cells")
plt.title("2001 Gi* z-score distribution with significance thresholds")

plt.legend()
plt.tight_layout()
plt.show()